In [ ]:
import time
import sys
import pandas as pd
import pickle

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ---- Decision Tree (DT) ----

# Dataset processing
df = pd.read_csv("dataset_packet_multiclass8_n1000000.csv", engine="pyarrow", index_col=0)

X = df.drop(["label"], axis=1)
y = df["label"]

(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)

# Model training
dtc = DecisionTreeClassifier()

start_time = time.process_time()
dtc.fit(X_train, y_train)
training_time = time.process_time() - start_time

# Model statistics
y_pred = dtc.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
full_report = classification_report(y_test, y_pred)

num_features = dtc.n_features_in_
size_kb = sys.getsizeof(pickle.dumps(dtc)) / 1024
depth = dtc.get_depth()
leaves = dtc.get_n_leaves()
param = dtc.get_params()

print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")
print(f"Depth            {depth}")
print(f"Leaves           {leaves}")
print(f"{'-'*53}")

print(f"Full Report\n{full_report}")
print(f"{'-'*53}")

print(f"Parameters:")
for p, v in param.items(): 
    print(f"{p:30}{v}")

In [ ]:
# Dataset processing
df = pd.read_csv("dataset_packet_multiclass8_n1000000.csv", engine="pyarrow", index_col=0)
(X, y) = (df.drop(["label"], axis=1), df["label"])
(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)

# Model training
dtc = DecisionTreeClassifier(
    criterion="gini",
    max_depth=50,             
    min_samples_leaf=100,   
    max_leaf_nodes=5000,   
    random_state=0
)

start_time = time.process_time()
dtc.fit(X_train, y_train)
training_time = time.process_time() - start_time

# Model statistics
y_pred = dtc.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
full_report = classification_report(y_test, y_pred)
num_features = dtc.n_features_in_
size_kb = sys.getsizeof(pickle.dumps(dtc)) / 1024

print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")


***
## Model Optimization

***
### Random Search with Cross Validation

In [ ]:
import time
import sys
import pandas as pd
import pickle

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from sklearn.model_selection import RandomizedSearchCV

# --- Random Serach using Packet Based Data ---

# Load the dataset
df = pd.read_csv("dataset_packet_multiclass8_n100000.csv", engine="pyarrow", index_col=0)
df.drop(["device_mac", "eth_src_oui", "eth_dst_oui"], axis=1) # Testing if some of the dataset features will make it unreasonable
(X, y) = (df.drop(["label"], axis=1), df["label"])
(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)

dtc = DecisionTreeClassifier(
    random_state=0,
)

param_distribution = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'splitter': ['best', 'random'],           # 'random' can significantly speed up training
    'max_depth': [None, 10, 20, 30, 50],      # Prevents the tree from becoming a "lookup table"
    'min_samples_split': [2, 10, 20, 50],     # Higher values prevent splitting on noise
    'min_samples_leaf': [1, 5, 10, 50],       # Essential for reducing RAM/Model size
    'max_features': [None, 'sqrt', 'log2'],   #  for e
    'ccp_alpha': [0.0, 0.001, 0.01, 0.1]      # Cost Complexity Pruning
}

dtc_random = RandomizedSearchCV(
    estimator=dtc, 
    param_distributions=param_distribution, 
    n_iter=50, 
    cv=5, 
    verbose=3, 
    random_state=42, 
    scoring='accuracy', 
    n_jobs=-1, 
    refit=True 
)

start_wall_time = time.perf_counter()
dtc_random.fit(X_train, y_train)
total_wall_time = time.perf_counter() - start_wall_time

best_dtc = dtc_random.best_estimator_

y_pred = best_dtc.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
size_kb = sys.getsizeof(pickle.dumps(best_dtc)) / 1024
training_time = dtc_random.cv_results_['mean_fit_time'][dtc_random.best_index_]

print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")

print(f"\n[+] Best Parameters: {dtc_random.best_params_}")
print(f"[+] Best Cross-Validation Accuracy: {dtc_random.best_score_:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Accuracy         0.9412
Precision        0.9412
Recall           0.9411
F1-Score         0.9411
Training Time    3.8977 s
Size in KB       938.5312 KB

[+] Best Parameters: {'splitter': 'random', 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 30, 'criterion': 'entropy', 'ccp_alpha': 0.0}
[+] Best Cross-Validation Accuracy: 0.9365


**--- n100000 ---**
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Accuracy         0.9412
Precision        0.9412
Recall           0.9411
F1-Score         0.9411
Training Time    2.6578 s
Size in KB       938.5312 KB

[+] Best Parameters: {'splitter': 'random', 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 30, 'criterion': 'entropy', 'ccp_alpha': 0.0}
[+] Best Cross-Validation Accuracy: 0.9365

**--- n1000000 ---**
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Accuracy         0.9766
Precision        0.9769
Recall           0.9766
F1-Score         0.9767
Training Time    61.5994 s
Size in KB       3141.2812 KB

[+] Best Parameters: {'splitter': 'random', 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 30, 'criterion': 'entropy', 'ccp_alpha': 0.0}
[+] Best Cross-Validation Accuracy: 0.9739


The best model parameters stayed the same accross all tests.

In [ ]:
import time
import sys
import pandas as pd
import pickle

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from sklearn.model_selection import RandomizedSearchCV

# --- Random Serach using Flow Based Data ---

# Load the dataset
df = pd.read_csv("dataset_flow_multiclass8_n100000.csv", engine="pyarrow", index_col=0)
(X, y) = (df.drop(["label"], axis=1), df["label"])
(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)

dtc = DecisionTreeClassifier(
    random_state=0,
)

param_distribution = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'splitter': ['best', 'random'],           # 'random' can significantly speed up training
    'max_depth': [None, 10, 20, 30, 50],      # Prevents the tree from becoming a "lookup table"
    'min_samples_split': [2, 10, 20, 50],     # Higher values prevent splitting on noise
    'min_samples_leaf': [1, 5, 10, 50],       # Essential for reducing RAM/Model size
    'max_features': [None, 'sqrt', 'log2'],   # Helps with 128-feature dimensionality
    'ccp_alpha': [0.0, 0.001, 0.01, 0.1]      # Cost Complexity Pruning
}

dtc_random = RandomizedSearchCV(
    estimator=dtc, 
    param_distributions=param_distribution, 
    n_iter=50, 
    cv=5, 
    verbose=3, 
    random_state=42, 
    scoring='accuracy', 
    n_jobs=-1, 
    refit=True 
)

start_wall_time = time.perf_counter()
dtc_random.fit(X_train, y_train)
total_wall_time = time.perf_counter() - start_wall_time

best_dtc = dtc_random.best_estimator_

y_pred = best_dtc.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
size_kb = sys.getsizeof(pickle.dumps(best_dtc)) / 1024
training_time = dtc_random.cv_results_['mean_fit_time'][dtc_random.best_index_]

print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")

print(f"\n[+] Best Parameters: {dtc_random.best_params_}")
print(f"[+] Best Cross-Validation Accuracy: {dtc_random.best_score_:.4f}")

**n100000**
Fitting 5 folds for each of 50 candidates, totalling 250 fits
Accuracy         0.7484
Precision        0.7313
Recall           0.7307
F1-Score         0.7309
Training Time    2.3614 s
Size in KB       4154.293 KB

[+] Best Parameters: {'splitter': 'random', 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 30, 'criterion': 'entropy', 'ccp_alpha': 0.0}
[+] Best Cross-Validation Accuracy: 0.7407

***
## Final Model Test

In [ ]:
import time
import sys
import pandas as pd
import pickle

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# ---- Decision Tree (DT) ----

# Dataset processing
# Load the dataset
df = pd.read_csv("dataset_packet_multiclass8_n1000000.csv", engine="pyarrow", index_col=0)
#df = pd.read_csv("dataset_packet_multiclass28_n100000.csv", engine="pyarrow", index_col=0) # All 28 classes. Accuracy drops to 0.9008. 

(X, y) = (df.drop(["label"], axis=1), df["label"])
(X_train, X_test, y_train, y_test) = train_test_split(X, y, train_size=0.8, random_state=0)

# Model training
dtc = DecisionTreeClassifier(
    random_state=0,
    splitter='random',
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=None,
    max_depth=30,
    criterion='entropy',
    ccp_alpha=0
)

start_time = time.process_time()
dtc.fit(X_train, y_train)
training_time = time.process_time() - start_time

# Model statistics
y_pred = dtc.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)
num_features = dtc.n_features_in_
size_kb = sys.getsizeof(pickle.dumps(dtc)) / 1024

print(f"Accuracy         {round(report["accuracy"], 4)}")
print(f"Precision        {round(report["macro avg"]["precision"], 4)}")
print(f"Recall           {round(report["macro avg"]["recall"], 4)}")
print(f"F1-Score         {round(report["macro avg"]["f1-score"], 4)}")
print(f"Training Time    {round(training_time, 4)} s")
print(f"Size in KB       {round(size_kb, 4)} KB")

with open("dtc_load_tester.pkl", 'wb') as file:
    pickle.dump(dtc, file)
print(f"Model successfully saved to dtc_load_tester.pkl")

Accuracy         0.9766
Precision        0.9769
Recall           0.9766
F1-Score         0.9767
Training Time    11.3906 s
Size in KB       3141.2744 KB
Model successfully saved to dtc_load_tester.pkl
